## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [2]:
sao_URL = "https://www.polyu.edu.hk/sao/"
start_idx, stop_idx = 36450, -900

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    return text

sao_loader = RecursiveUrlLoader(
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"news-and-events",
        sao_URL+"News-and-Events", 
        sao_URL+"Sitemap", 
        sao_URL+"Search-Result",
        sao_URL+"National-Education",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"Student-Development-Section",
        sao_URL+"student-development-section",
        sao_URL+"Counselling-and-Wellness-Section/PolyU-Asian-Universities-Water-Polo-Invitational-Tournament",
        sao_URL+"Counselling-and-Wellness-Section/Wellness-Centre",
        sao_URL+"Counselling-and-Wellness-Section/Sports-Development",
        sao_URL+"Counselling-and-Wellness-Section/Programmes-and-Activities",
        sao_URL+"Student-Resources-and-Support-Section/Outstanding-Student-Academy",
        sao_URL+"Careers-and-Placement-Section/Gallery-and-Publications",
        sao_URL+"Non-local-Student-Services/Event-Highlight",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [3]:
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(doc)

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 230
page_content='      Mandatory Online Training Module on Preventing Sexual Harassment on Campus
                            

                                Contact Us
                            

Quick Access

Start main content

													Home
												

													Counselling and Wellness
												

													Student Counselling
												

													Resources
												

													Student Adjustment (for new student)
												

Student Adjustment (for new student)

Adjustment issues are expected and common

No matter whether you are a first year student, a senior year admitted student, a new postgraduate or a student on exchange, it is very normal for you to experience some adjustment issues as the system and practices at PolyU could be quite different from where you were previously. Also, you may have to meet up with many new faces, follow up with many tasks and participate in many activities within a short perio

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"

In [5]:
print(chunks[10])

page_content='Application for Hall Residence 2025/26 – Full-time undergraduate students

Student Type

Application Period

Current students
(including Special Readmission Scheme (SRS) &
Readmission Scheme of CURI Residential College (RSCRC))
2 May (10:00am) to 19 May 2025 (11:59pm)

Non-local new students

11 Jun (10:00am) - 25 Jul 2025 (11:59pm) - Phase 1
             26 Jul - 29 Aug 2025 (11:59pm) - Phase 2

Local new students
Stage (1): 11 Aug (10:00am) – 15 Aug 2025 (11:59pm) 
            Stage (2): 16 Aug (10:00am) - 4 Sep 2025 (11:59pm)

Inbound exchange students of Semester 1*

Mid-Jul 2026

Inbound exchange students of Semester 2*

2 Dec 2025 (10:00am) - 9 Dec 2025 (11:59pm)

*Global Engagement Office will inform eligible inbound exchange students the application and arrangement of hall accommodation by email in due course.
Application for Summer Hall Residence 2026 – Full-time undergraduate students

Student Type

Application Period

Full-time undergraduate students
Mid-Apr 20

### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_sao_webpage" if not SINGLE else "vaa_documents"

In [7]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_17758/116003828.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


123

In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i+ 500)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 1094 chunks into ChromaDB to vaa_documents


/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_17758/3474824168.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


### 4. Simple Testing

In [9]:
query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: Admission Policies for Undergraduates

Admission Policy

General Information on Application

Special Readmission Scheme (SRS)

Readmission Scheme of CURI Residential College (RSCRC)

 
Admission Policy
A. Eligibility
The PolyU Student Halls were established with major funding support from the University Grants Committee (UGC), hence, eligibility to hall residence has to be set with reference to the UGC guidelines. The University has come up with a set of policies to govern admission of students to hall residence. The following groups of students are eligible for hall residence:

Full-time Research Students within normal study period (Admission Policy of Student Halls for Research Students)
Full-time Local Students in UGC-funded Bachelor’s Degree Programmes
Full-time Non-local students in UGC-funded Bachelor’s Degree Programmes
Full-time Inbound Exchange Students in UGC-funded Bachelor’s Degree Programmes...
Source: https://www.polyu.edu.hk/sao/student-resources-and-support-sec